# Evaluacion del agente entrenado

Carga el modelo de competencia, corre los episodios de evaluacion con politica greedy grabando video de cada uno, reporta el puntaje de cada episodio junto con su video, y embebe todos los videos para reproducirlos dentro del notebook. Pensado para correr celda por celda el dia de la presentacion, sin pasos manuales.

Reutiliza las funciones de `src/evaluar.py` (que a su vez reusan `src/ale_utils.py`): `cargar_modelo`, `generar_video` y `predecir` (politica greedy, `deterministic=True`).

In [ ]:
import sys, os

for candidato in [".", "..", "../.."]:
    if os.path.isdir(os.path.join(candidato, "src")):
        RAIZ = os.path.abspath(candidato)
        break
else:
    raise RuntimeError("no se encontro el directorio src del proyecto")

sys.path.append(os.path.join(RAIZ, "src"))

import glob as glob_mod
import numpy as np
from IPython.display import Video, display

import evaluar
print(f"raiz del proyecto: {RAIZ}")

## Configuracion

Modelo de competencia por defecto. Se puede apuntar `RUTA_MODELO` a cualquier otro run con `config.json` junto al `.zip`.

In [ ]:
RUTA_MODELO = os.path.join(RAIZ, "modelos/competencia/modelo_competencia.zip")
N_EPISODIOS = 5
SEMILLA = 42

run_id = os.path.basename(os.path.dirname(RUTA_MODELO))
carpeta_video = os.path.join(RAIZ, f"modelos/videos/{run_id}")
print(f"modelo: {RUTA_MODELO}  (run_id={run_id})")
print(f"episodios: {N_EPISODIOS}  |  semilla: {SEMILLA}")
print(f"carpeta de video: {carpeta_video}")

## Cargar modelo

`cargar_modelo` lee `config.json` junto al `.zip` para reconstruir la arquitectura (extractor, dueling, algoritmo) y los pesos.

In [ ]:
modelo = evaluar.cargar_modelo(RUTA_MODELO)
config = evaluar.leer_config(RUTA_MODELO)
escala_grises = config.get("escala_grises", True)

print(f"algoritmo: {config.get('algoritmo')}")
print(f"backbone: {config.get('backbone')}")
print(f"dueling: {config.get('dueling', False)}")
print(f"escala_grises: {escala_grises}")
print(f"n_apilados: {config.get('n_apilados', 4)}")
print(f"pasos entrenados: {modelo.num_timesteps}")

## Evaluacion greedy con video

Corre los `N_EPISODIOS` episodios con politica greedy (`deterministic=True`), recompensa real del juego sin clipping, grabando video de cada episodio. Al terminar imprime el puntaje de cada video y el resumen (max / media).

In [ ]:
os.makedirs(carpeta_video, exist_ok=True)
for viejo in glob_mod.glob(os.path.join(carpeta_video, "*.mp4")):
    os.remove(viejo)

rutas, puntajes = evaluar.generar_video(
    modelo,
    carpeta_video,
    n_episodios=N_EPISODIOS,
    semilla=SEMILLA,
    escala_grises=escala_grises,
)

print()
for ruta, puntaje in zip(rutas, puntajes):
    print(f"{os.path.basename(ruta)}: {puntaje:.0f} puntos")
print(f"\nmax {max(puntajes):.0f} / media {np.mean(puntajes):.1f}")

## Videos embebidos

Cada video con su puntaje, embebido para reproducirlo dentro del notebook durante la presentacion.

In [ ]:
for ruta, puntaje in zip(rutas, puntajes):
    print(f"{os.path.basename(ruta)}: {puntaje:.0f} puntos")
    display(Video(ruta, embed=True))